# SRSDB Clarity Notebook

Use this notebook to explore `srsdb` quickly: schemas, tables, columns, constraints, indexes, row samples, and custom SQL.

## 1) Install dependencies (run once)
If these packages are already installed, this cell is harmless.

In [1]:
%pip install -q pandas sqlalchemy psycopg[binary] ipywidgets


Note: you may need to restart the kernel to use updated packages.


## 2) Configure database connection
Defaults match your project MCP config (`127.0.0.1:5434`, `srsdb`, `postgres/postgres`).

In [2]:
import os
from sqlalchemy import create_engine, text
import pandas as pd

DB_HOST = os.getenv('SRSDB_HOST', '127.0.0.1')
DB_PORT = int(os.getenv('SRSDB_PORT', '5434'))
DB_NAME = os.getenv('SRSDB_NAME', 'srsdb')
DB_USER = os.getenv('SRSDB_USER', 'postgres')
DB_PASSWORD = os.getenv('SRSDB_PASSWORD', 'postgres')

DATABASE_URL = f'postgresql+psycopg://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(DATABASE_URL, pool_pre_ping=True)
print(f'Configured for {DB_HOST}:{DB_PORT}/{DB_NAME} as {DB_USER}')


Configured for 127.0.0.1:5434/srsdb as postgres


In [3]:
with engine.connect() as conn:
    row = conn.execute(text("SELECT current_database(), current_user, version()")).fetchone()
print(row)


('srsdb', 'postgres', 'PostgreSQL 16.13 on x86_64-pc-linux-musl, compiled by gcc (Alpine 15.2.0) 15.2.0, 64-bit')


## 3) Reusable helpers

In [4]:
def run_sql(sql: str, params: dict | None = None, max_rows: int | None = 200) -> pd.DataFrame:
    with engine.connect() as conn:
        df = pd.read_sql(text(sql), conn, params=params or {})
    if max_rows is not None and len(df) > max_rows:
        return df.head(max_rows)
    return df

def show_schemas() -> pd.DataFrame:
    return run_sql("""
    SELECT schema_name
    FROM information_schema.schemata
    ORDER BY schema_name
    """)

def show_tables(schema: str) -> pd.DataFrame:
    return run_sql("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_type = 'BASE TABLE'
      AND table_schema = :schema
    ORDER BY table_name
    """, {"schema": schema})

def show_columns(schema: str, table: str) -> pd.DataFrame:
    return run_sql("""
    SELECT column_name, data_type, is_nullable, character_maximum_length, column_default
    FROM information_schema.columns
    WHERE table_schema = :schema AND table_name = :table
    ORDER BY ordinal_position
    """, {"schema": schema, "table": table})

def preview_table(schema: str, table: str, limit: int = 10) -> pd.DataFrame:
    safe = f'"{schema}"."{table}"'
    return run_sql(f'SELECT * FROM {safe} LIMIT {int(limit)}', max_rows=None)

def find_tables(keyword: str) -> pd.DataFrame:
    return run_sql("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_type = 'BASE TABLE'
      AND (table_name ILIKE :kw OR table_schema ILIKE :kw)
    ORDER BY table_schema, table_name
    """, {"kw": f'%{keyword}%'})

def find_columns(keyword: str) -> pd.DataFrame:
    return run_sql("""
    SELECT table_schema, table_name, column_name, data_type
    FROM information_schema.columns
    WHERE column_name ILIKE :kw
    ORDER BY table_schema, table_name, ordinal_position
    """, {"kw": f'%{keyword}%'})

def table_constraints(schema: str, table: str) -> pd.DataFrame:
    return run_sql("""
    SELECT tc.constraint_name, tc.constraint_type, kcu.column_name
    FROM information_schema.table_constraints tc
    LEFT JOIN information_schema.key_column_usage kcu
      ON tc.constraint_name = kcu.constraint_name
     AND tc.table_schema = kcu.table_schema
     AND tc.table_name = kcu.table_name
    WHERE tc.table_schema = :schema AND tc.table_name = :table
    ORDER BY tc.constraint_type, tc.constraint_name
    """, {"schema": schema, "table": table})

def table_indexes(schema: str, table: str) -> pd.DataFrame:
    return run_sql("""
    SELECT schemaname, tablename, indexname, indexdef
    FROM pg_indexes
    WHERE schemaname = :schema AND tablename = :table
    ORDER BY indexname
    """, {"schema": schema, "table": table})


## 4) Quick discovery (recommended first)

Run these cells to understand the whole database quickly.

In [5]:
show_schemas()


,schema_name
0,information_schema
1,pg_catalog
2,pg_toast
3,public
4,srsadmin


In [6]:
show_tables('srsadmin')


,table_schema,table_name
0,srsadmin,rc_beneficiary
1,srsadmin,rc_beneficiary_303
2,srsadmin,rc_beneficiary_304
3,srsadmin,rc_beneficiary_305
4,srsadmin,rc_beneficiary_306
...,...,...
120,srsadmin,swasthya_sathi_transaction_2526_321
121,srsadmin,swasthya_sathi_transaction_2526_664
122,srsadmin,swasthya_sathi_transaction_2526_702
123,srsadmin,swasthya_sathi_transaction_2526_703


In [7]:
show_tables('public')


,table_schema,table_name
0,public,analyses
1,public,audit_alerts
2,public,audit_cases
3,public,case_action_logs
4,public,citizen_enrollments
...,...,...
65,public,swasthya_sathi_transaction_2526_321
66,public,swasthya_sathi_transaction_2526_664
67,public,swasthya_sathi_transaction_2526_702
68,public,swasthya_sathi_transaction_2526_703


In [8]:
find_columns('fullname')


,table_schema,table_name,column_name,data_type
0,public,citizen_profiles,fullname,character varying
1,public,swasthya_sathi_beneficiary,fullname,character varying
2,public,swasthya_sathi_beneficiary_000,fullname,character varying
3,public,swasthya_sathi_beneficiary_303,fullname,character varying
4,public,swasthya_sathi_beneficiary_304,fullname,character varying
...,...,...,...,...
96,srsadmin,swasthya_sathi_beneficiary_321,fullname,character varying
97,srsadmin,swasthya_sathi_beneficiary_664,fullname,character varying
98,srsadmin,swasthya_sathi_beneficiary_702,fullname,character varying
99,srsadmin,swasthya_sathi_beneficiary_703,fullname,character varying


In [9]:
show_columns('public', 'swasthya_sathi_beneficiary')


,column_name,data_type,is_nullable,character_maximum_length,column_default
0,sl_no,bigint,NO,NaN,nextval('swasthya_sathi_beneficiary_sl_no_seq'...
1,srs_ben_code,integer,YES,NaN,None
2,dept_id,character varying,NO,2.0,None
3,scheme_id,character varying,NO,4.0,None
4,scheme_beneficiary_id,character varying,NO,30.0,None
5,uid,character varying,YES,12.0,None
6,ration_card_number,character varying,YES,30.0,None
7,ration_card_memberid,character varying,YES,15.0,None
8,fullname,character varying,YES,200.0,None
9,member_dob,character varying,YES,15.0,None


In [ ]:
table_constraints('public', 'swasthya_sathi_beneficiary')


In [ ]:
table_indexes('public', 'swasthya_sathi_beneficiary').head(20)


In [ ]:
preview_table('public', 'swasthya_sathi_beneficiary', limit=10)


## 5) Ask your own SQL questions

Edit and run this cell repeatedly.

In [14]:
sql = """
SELECT table_schema, table_name
FROM information_schema.tables
WHERE table_schema IN ('srsadmin')
ORDER BY table_schema, table_name
LIMIT 10;
"""
run_sql(sql, max_rows=None)


,table_schema,table_name
0,srsadmin,rc_beneficiary
1,srsadmin,rc_beneficiary_303
2,srsadmin,rc_beneficiary_304
3,srsadmin,rc_beneficiary_305
4,srsadmin,rc_beneficiary_306
5,srsadmin,rc_beneficiary_307
6,srsadmin,rc_beneficiary_308
7,srsadmin,rc_beneficiary_309
8,srsadmin,rc_beneficiary_310
9,srsadmin,rc_beneficiary_311
